In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [ ]:
DB_NAME = "vector_db"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

# Company RAG

Answers questions from the documents in `knowledge-base/`.

1. Load and chunk the documents
2. Embed the chunks
3. Retrieve the chunks closest to the question
4. Ask the model to answer using only those chunks

In [ ]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
KB_DIR = Path("knowledge-base")

## 1. Load and chunk

In [ ]:
def chunk(text, size=800, overlap=100):
    pieces = []
    start = 0
    while start < len(text):
        pieces.append(text[start:start + size].strip())
        start += size - overlap
    return [p for p in pieces if p]


chunks = []
for path in sorted(KB_DIR.rglob("*.md")):
    for piece in chunk(path.read_text(encoding="utf-8")):
        chunks.append({"source": path.name, "text": piece})

print(f"{len(chunks)} chunks from {len({c['source'] for c in chunks})} documents")

## 2. Embed

In [ ]:
def embed(texts):
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vectors = np.array([d.embedding for d in response.data])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


index = embed([c["text"] for c in chunks])
index.shape

## 3. Retrieve

In [ ]:
def retrieve(question, k=3):
    scores = index @ embed([question])[0]
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]


for hit in retrieve("How many days of leave do I get?"):
    print(f"{hit['score']:.3f}  {hit['source']}")

## 4. Answer

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions about the company using only the context provided. "
    "If the context does not contain the answer, say you don't know."
)


def ask(question, k=3):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

In [ ]:
print(ask("How many days of leave do I get?"))

In [ ]:
print(ask("What does StockPilot cost?"))